### Transform Circuits Data

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

#### Step 1 - Read Bronze Circuits Table

In [0]:
circuits_df = spark.table(bronze_table)

#### Step 2 - Keep only the column required for Analytics(Drop url column)

In [0]:
from pyspark.sql import functions as F

In [0]:
circuits_selected_df = circuits_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("ingestion_timestamp"),
    F.col("source")
)

#### Step 3 & 4 - Standardize Column Names
- Standardise column names in snake_case(circuitId -> circuit_id, circuitName -> circuit_name)
- Rename columns to make them more meaningful(lat -> latitude, long -> longitude)

In [0]:
circuits_renamed_df = circuits_selected_df.withColumnsRenamed({
    "circuitId" : "circuit_id",
    "circuitName" : "circuit_name",
    "lat" : "latitude",
    "long" : "longitude"
})

#### Step 5 - Filter out rows where circuit_id is NULL(business key validation)

In [0]:
circuits_valid_df = circuits_renamed_df.filter(
    F.col("circuit_id").isNotNull()
)

#### Step 6 - Remove Duplicate Records

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])

#### Step 7 - Transform values of columns circuit_name and locality to Title Case

In [0]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn('circuit_name', F.initcap(F.col("circuit_name")))
        .withColumn('locality', F.initcap(F.col("locality")))
)

#### Step 8 - Write the transformed data into silver circuits table

In [0]:
(
    circuits_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table)
)